# Advanced Mappings: Copy, Tree, and Diff

## Overview

Master advanced mapping operations for complex integration scenarios. This notebook covers utilities that help you manage mapping evolution, compare versions, and understand resource hierarchies.

**What you'll learn:**
- Copy mappings with `copy()` for templates or testing
- View mapping hierarchies with `get_tree()` (versions → snapshots → instances)
- Compare mapping versions with `diff()` to track schema changes

**Prerequisites:**
- Completed: [Analyst tutorials](../analyst/) (recommended)
- Understanding of mappings, snapshots, and instances

**Time to complete:** 20 minutes

---

**Test Coverage:**
- Copy mapping functionality
- Mapping hierarchy tree retrieval
- Version diff between mapping versions

In [ ]:
import os

# Parameters cell - papermill will inject values here
# Note: Uses GRAPH_OLAP_API_URL from environment (set by JupyterHub or local dev)
SEEDED_MAPPING_ID = None  # Injected by papermill from fixtures

## Setup

### Test Data Strategy

This test creates mappings with multiple versions to test advanced features:
1. **Base Mapping**: Created with initial version
2. **Updated Mapping**: Create version 2 for diff testing
3. **Copied Mapping**: Test copy functionality
4. **Snapshots**: Create snapshots to test hierarchy tree
5. **Cleanup**: Automatic cleanup via cleanup framework

In [ ]:
import sys
import os

print(f"Python version: {sys.version}")
print(f"GRAPH_OLAP_API_URL: {os.environ.get('GRAPH_OLAP_API_URL', 'not set')}")

In [ ]:
from graph_olap import notebook
from graph_olap.exceptions import NotFoundError
from graph_olap.models.mapping import EdgeDefinition, NodeDefinition, PropertyDefinition
from graph_olap.testing import TestPersona
from graph_olap_schemas import WrapperType

print("SDK imports successful")

# Wake up Starburst Galaxy cluster (auto-suspends after 5 min idle)
notebook.wake_starburst()

## Connect to SDK

In [ ]:
# Create test context with automatic cleanup
ctx = notebook.test("MappingAdvTest", persona=TestPersona.ANALYST_ALICE)
client = ctx.client

print(f"Connected to {client._config.api_url}")
print(f"Test run ID: {ctx.run_id}")

## Initialize Cleanup Tracking

In [ ]:
# Resources are automatically tracked and cleaned up via ctx
print("Starting Mapping Advanced Features E2E Test - resources will be cleaned up automatically via atexit")

## Create Base Mapping

Create a mapping with initial version for testing advanced features.

In [ ]:
# Create base mapping using ctx.mapping (auto-tracked)
person_node = NodeDefinition(
    label="Person",
    sql="SELECT DISTINCT id, name, age FROM bigquery.graph_olap_e2e.persons",
    primary_key={"name": "id", "type": "STRING"},
    properties=[
        PropertyDefinition(name="name", type="STRING"),
        PropertyDefinition(name="age", type="INT64"),
    ]
)

knows_edge = EdgeDefinition(
    type="KNOWS",
    from_node="Person",
    to_node="Person",
    sql="SELECT DISTINCT from_id, to_id, since FROM bigquery.graph_olap_e2e.friendships",
    from_key="from_id",
    to_key="to_id",
    properties=[
        PropertyDefinition(name="since", type="INT64"),
    ]
)

base_mapping = ctx.mapping(
    description="Base mapping for advanced features test",
    node_definitions=[person_node],
    edge_definitions=[knows_edge],
)
BASE_MAPPING_ID = base_mapping.id
BASE_MAPPING_NAME = base_mapping.name

print(f"Created base mapping: {BASE_MAPPING_NAME} (id={BASE_MAPPING_ID}, version={base_mapping.current_version})")

## 1. Test Mapping Copy

### 1.1 Test copy() Creates New Mapping

In [ ]:
# Test: Copy mapping
COPY_NAME = f"MappingAdvTest-Copy-{ctx.run_id}"

copied_mapping = client.mappings.copy(BASE_MAPPING_ID, new_name=COPY_NAME)

assert copied_mapping is not None, "Copied mapping should not be None"
assert copied_mapping.id != BASE_MAPPING_ID, "Copy should have different ID"
assert copied_mapping.name == COPY_NAME, f"Expected name '{COPY_NAME}', got '{copied_mapping.name}'"
assert copied_mapping.current_version == 1, f"Copy should start at version 1, got {copied_mapping.current_version}"

COPIED_MAPPING_ID = copied_mapping.id
# Track copied mapping for cleanup via ctx
ctx._track('mapping', COPIED_MAPPING_ID, COPY_NAME)

print(f"MAPPING_ADV 1.1 PASSED: Copied mapping {BASE_MAPPING_ID} → {COPIED_MAPPING_ID}")
print(f"  Original: {base_mapping.name}")
print(f"  Copy: {copied_mapping.name}")

### 1.2 Test Copy Has Same Schema

In [ ]:
# Test: Verify copy has same node and edge definitions
assert len(copied_mapping.node_definitions) == len(base_mapping.node_definitions), \
    "Copy should have same number of node definitions"
assert len(copied_mapping.edge_definitions) == len(base_mapping.edge_definitions), \
    "Copy should have same number of edge definitions"

# Verify node labels match
base_node_labels = {n.label for n in base_mapping.node_definitions}
copied_node_labels = {n.label for n in copied_mapping.node_definitions}
assert base_node_labels == copied_node_labels, "Node labels should match"

# Verify edge types match
base_edge_types = {e.type for e in base_mapping.edge_definitions}
copied_edge_types = {e.type for e in copied_mapping.edge_definitions}
assert base_edge_types == copied_edge_types, "Edge types should match"

print("MAPPING_ADV 1.2 PASSED: Copy has identical schema")
print(f"  Node labels: {copied_node_labels}")
print(f"  Edge types: {copied_edge_types}")

### 1.3 Test Copy is Independent

In [ ]:
# Test: Modifying copy doesn't affect original
company_node = NodeDefinition(
    label="Company",
    sql="SELECT 'ACME' as id, 'ACME Corp' as name",
    primary_key={"name": "id", "type": "STRING"},
    properties=[PropertyDefinition(name="name", type="STRING")],
)

updated_copy = client.mappings.update(
    COPIED_MAPPING_ID,
    change_description="Add Company node to copy",
    node_definitions=copied_mapping.node_definitions + [company_node],
    edge_definitions=copied_mapping.edge_definitions,
)

assert updated_copy.current_version == 2, "Copy should be at version 2"

# Verify original is unchanged
original_fresh = client.mappings.get(BASE_MAPPING_ID)
assert original_fresh.current_version == 1, "Original should still be at version 1"
assert len(original_fresh.node_definitions) == 1, "Original should still have 1 node"

print("MAPPING_ADV 1.3 PASSED: Copy is independent")
print(f"  Copy version: {updated_copy.current_version} (2 nodes)")
print(f"  Original version: {original_fresh.current_version} (1 node)")

## 2. Test Mapping Hierarchy Tree

### 2.1 Test get_tree() Returns Hierarchy

In [ ]:
# Test: Get mapping tree (version → snapshot → instance hierarchy)
tree = client.mappings.get_tree(BASE_MAPPING_ID)

assert tree is not None, "Tree should not be None"
assert isinstance(tree, dict), f"Tree should be dict, got {type(tree)}"
assert 1 in tree, "Tree should contain version 1"

# Version 1 should exist
version_1 = tree[1]
assert "name" in version_1, "Version should have name"
assert "snapshots" in version_1, "Version should have snapshots list"

print("MAPPING_ADV 2.1 PASSED: get_tree() returned hierarchy")
print(f"  Versions: {list(tree.keys())}")
print(f"  Version 1: {version_1['name']}")

### 2.2 Test Tree With Snapshots

In [ ]:
# Create a snapshot to populate the tree
SNAPSHOT_NAME = f"MappingAdvTest-Snapshot-{ctx.run_id}"

snapshot = client.snapshots.create(
    mapping_id=BASE_MAPPING_ID,
    name=SNAPSHOT_NAME,
)
SNAPSHOT_ID = snapshot.id
# Track snapshot for cleanup via ctx
ctx._track('snapshot', SNAPSHOT_ID, SNAPSHOT_NAME)

print(f"Created snapshot: {SNAPSHOT_NAME} (id={SNAPSHOT_ID})")

# Get tree again (should now include snapshot)
tree_with_snapshot = client.mappings.get_tree(BASE_MAPPING_ID, include_instances=False)

version_1 = tree_with_snapshot[1]
assert len(version_1["snapshots"]) >= 1, "Version 1 should have at least 1 snapshot"

# Find our snapshot
our_snapshot = next((s for s in version_1["snapshots"] if s["id"] == SNAPSHOT_ID), None)
assert our_snapshot is not None, f"Snapshot {SNAPSHOT_ID} should be in tree"
assert our_snapshot["name"] == SNAPSHOT_NAME

print("MAPPING_ADV 2.2 PASSED: Tree includes snapshots")
print(f"  Version 1 snapshots: {len(version_1['snapshots'])}")

### 2.3 Test Tree With Instances

In [ ]:
# Create an instance to add to the tree
INSTANCE_NAME = f"MappingAdvTest-Instance-{ctx.run_id}"

# Wait for snapshot to be ready first
import time

timeout = 120
start = time.time()
while time.time() - start < timeout:
    snapshot = client.snapshots.get(SNAPSHOT_ID)
    if snapshot.status == "ready":
        break
    elif snapshot.status == "failed":
        print("Snapshot failed, skipping instance creation")
        break
    time.sleep(3)

if snapshot.status == "ready":
    instance = client.instances.create(
        snapshot_id=SNAPSHOT_ID,
        name=INSTANCE_NAME,
        wrapper_type=WrapperType.RYUGRAPH,
    )
    INSTANCE_ID = instance.id
    # Track instance for cleanup via ctx
    ctx._track('instance', INSTANCE_ID, INSTANCE_NAME)

    print(f"Created instance: {INSTANCE_NAME} (id={INSTANCE_ID})")

    # Get tree with instances
    tree_full = client.mappings.get_tree(BASE_MAPPING_ID, include_instances=True)

    # Find our snapshot and verify it has instances
    version_1 = tree_full[1]
    our_snapshot = next((s for s in version_1["snapshots"] if s["id"] == SNAPSHOT_ID), None)
    assert our_snapshot is not None
    assert "instances" in our_snapshot, "Snapshot should have instances list"

    # Verify our instance is in the tree
    our_instance = next((i for i in our_snapshot["instances"] if i["id"] == INSTANCE_ID), None)
    assert our_instance is not None, f"Instance {INSTANCE_ID} should be in tree"
    assert our_instance["name"] == INSTANCE_NAME

    print("MAPPING_ADV 2.3 PASSED: Tree includes instances")
    print(f"  Snapshot {SNAPSHOT_ID} instances: {len(our_snapshot['instances'])}")
else:
    print(f"MAPPING_ADV 2.3 SKIPPED: Snapshot not ready (status={snapshot.status})")

## 3. Test Version Diff

### 3.1 Create Version 2 for Diff Testing

In [ ]:
# Update base mapping to create version 2
print(f"BASE_MAPPING_ID={BASE_MAPPING_ID}")

# Fetch current state first to ensure we have latest definitions
current_base = client.mappings.get(BASE_MAPPING_ID)
print(f"Current base mapping: id={current_base.id}, version={current_base.current_version}, name={current_base.name}")
print(f"  Nodes: {[n.label for n in current_base.node_definitions]}")
print(f"  Edges: {[e.type for e in current_base.edge_definitions]}")

# Add Location node
location_node = NodeDefinition(
    label="Location",
    sql="SELECT 'NYC' as id, 'New York City' as name",
    primary_key={"name": "id", "type": "STRING"},
    properties=[PropertyDefinition(name="name", type="STRING")],
)

print(f"\nUpdating with {len(current_base.node_definitions) + 1} nodes...")
updated_mapping = client.mappings.update(
    BASE_MAPPING_ID,
    change_description="Add Location node for diff test",
    node_definitions=current_base.node_definitions + [location_node],
    edge_definitions=current_base.edge_definitions,
)

print(f"\nAfter update: id={updated_mapping.id}, version={updated_mapping.current_version}")
print(f"  Nodes: {[n.label for n in updated_mapping.node_definitions]}")

assert updated_mapping.id == BASE_MAPPING_ID, f"Mapping ID changed: {BASE_MAPPING_ID} → {updated_mapping.id}"
assert updated_mapping.current_version == 2, f"Expected version 2, got {updated_mapping.current_version}"

print("\n✓ Successfully created version 2")
print(f"  Version 1: {len(current_base.node_definitions)} node(s), {len(current_base.edge_definitions)} edge(s)")
print(f"  Version 2: {len(updated_mapping.node_definitions)} node(s), {len(updated_mapping.edge_definitions)} edge(s)")

### 3.2 Test diff_versions() Shows Changes

In [ ]:
# Test: Diff versions 1 and 2
diff = client.mappings.diff(BASE_MAPPING_ID, from_version=1, to_version=2)

assert diff is not None, "Diff should not be None"

# Verify diff structure
assert hasattr(diff, 'summary'), "Diff should have summary"
assert hasattr(diff, 'changes'), "Diff should have changes"

# Verify summary shows the added Location node
assert diff.summary["nodes_added"] == 1, f"Expected 1 node added, got {diff.summary['nodes_added']}"
assert diff.summary["nodes_removed"] == 0, f"Expected 0 nodes removed, got {diff.summary['nodes_removed']}"
assert diff.summary["nodes_modified"] == 0, f"Expected 0 nodes modified, got {diff.summary['nodes_modified']}"

# Verify changes list contains Location node
added_nodes = [n for n in diff.changes["nodes"] if n.change_type == "added"]
assert len(added_nodes) == 1, f"Expected 1 added node in changes, got {len(added_nodes)}"
assert added_nodes[0].label == "Location", f"Expected Location node, got {added_nodes[0].label}"

print("MAPPING_ADV 3.2 PASSED: diff() shows changes correctly")
print(f"  Summary: +{diff.summary['nodes_added']} nodes, -{diff.summary['nodes_removed']} nodes")
print(f"  Added node: {added_nodes[0].label}")

### 3.3 Test Diff From Version 2 to 1 (Reverse)

In [ ]:
# Test: Reverse diff (2 → 1 should show removed node)
reverse_diff = client.mappings.diff(BASE_MAPPING_ID, from_version=2, to_version=1)

assert reverse_diff is not None

# Verify reverse diff shows opposite changes
assert reverse_diff.summary["nodes_added"] == 0, f"Expected 0 nodes added, got {reverse_diff.summary['nodes_added']}"
assert reverse_diff.summary["nodes_removed"] == 1, f"Expected 1 node removed, got {reverse_diff.summary['nodes_removed']}"
assert reverse_diff.summary["nodes_modified"] == 0, f"Expected 0 nodes modified, got {reverse_diff.summary['nodes_modified']}"

# Verify changes list contains Location node as removed
removed_nodes = [n for n in reverse_diff.changes["nodes"] if n.change_type == "removed"]
assert len(removed_nodes) == 1, f"Expected 1 removed node in changes, got {len(removed_nodes)}"
assert removed_nodes[0].label == "Location", f"Expected Location node, got {removed_nodes[0].label}"

print("MAPPING_ADV 3.3 PASSED: Reverse diff works correctly")
print(f"  Forward (1→2): +{diff.summary['nodes_added']} nodes")
print(f"  Reverse (2→1): -{reverse_diff.summary['nodes_removed']} nodes")

## Cleanup

In [ ]:
# Cleanup is handled automatically by ctx via atexit
# For interactive use, you can call ctx.cleanup() manually

print("\nCleanup will happen automatically via atexit")

## Summary

In [ ]:
print("\n" + "="*60)
print("MAPPING ADVANCED FEATURES E2E TESTS COMPLETED!")
print("="*60)
print("\nValidated:")
print("  1. Mapping Copy:")
print("    1.1: copy() creates new mapping")
print("    1.2: Copy has identical schema")
print("    1.3: Copy is independent of original")
print("  2. Mapping Hierarchy Tree:")
print("    2.1: get_tree() returns version hierarchy")
print("    2.2: Tree includes snapshots")
print("    2.3: Tree includes instances (when include_instances=True)")
print("  3. Version Diff:")
print("    3.1: Create version 2 for testing")
print("    3.2: diff_versions() shows changes between versions")
print("    3.3: Reverse diff works (from newer to older)")
print("\nAll resources will be cleaned up automatically via atexit")